# 🎓 EduMIND — Multi-Task LoRA Fine-Tuning LLM (Semantic Alignment & Q&A)

> **Dataset**: `corpus.jsonl` — Generated multi-task instruction tuning dataset
> **Framework**: [UnSloth](https://github.com/unslothai/unsloth) + PEFT LoRA
> **Target**: Fine-tune a causal LLM on two tasks: (1) VietMix-to-English translation & (2) Domain Q&A pedagogical responses
> **Hardware**: Kaggle 1 NVIDIA T4

---

## 🗺️ Notebook Structure

1. **Environment Setup** — Install UnSloth, verify GPUs
2. **Dataset Loading** — Load local or Kaggle input `corpus.jsonl`
3. **Data Preprocessing** — Build multi-task ChatML prompt formatting
4. **Model Loading** — Load base LLM with 4-bit quantization (QLoRA)
5. **LoRA Config** — Apply parameter-efficient fine-tuning adapters
6. **Training** — Fine-tune with `SFTTrainer` (multi-GPU via DDP)
7. **Evaluation** — BLEU on Task 1, qualitative inspection on Task 2
8. **Save & Export** — Merge adapters and push to HuggingFace Hub / GGUF

## 1️⃣ Environment Setup


In [ ]:
# ── Detect hardware ──────────────────────────────────────────────────────────
import subprocess
import sys

gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu_info.stdout)

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Count GPUs
import torch
NUM_GPUS = torch.cuda.device_count()
print(f"\n✅ Found {NUM_GPUS} GPU(s)")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")

In [ ]:
# ── Install UnSloth + dependencies ─────────────────────────────────────────
# UnSloth optimizes memory & speed for LoRA fine-tuning on consumer GPUs
!pip install --quiet "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet datasets peft accelerate bitsandbytes sacrebleu evaluate huggingface_hub

print("✅ Unsloth default packages successfully installed without conflict.")

In [ ]:
# ── Global Configuration ─────────────────────────────────────────────────────
import os
import torch

# ── Model choice (change if you want a different base LLM) ──────────────────
# Recommended choices for 2×T4 (15 GB VRAM each):
#   - "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   (best quality, fits 2×T4)
#   - "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
#   - "unsloth/llama-3-8b-Instruct-bnb-4bit"
#   - "unsloth/gemma-2-9b-it-bnb-4bit"          (slightly slower)
BASE_MODEL_ID   = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ── Output paths ────────────────────────────────────────────────────────────
OUTPUT_DIR      = "/kaggle/working/edumind-vietmix-llm"
HF_REPO_ID      = "Khang3004/edumind-vietmix-qwen2.5-7b-lora"  # ← change this

# ── Training hyperparameters ─────────────────────────────────────────────────
MAX_SEQ_LEN     = 512      # max input+output token length
BATCH_SIZE      = 2        # per-device batch size
GRAD_ACCUM      = 8        # effective batch = BATCH_SIZE * GRAD_ACCUM * NUM_GPUS
EPOCHS          = 3
LEARNING_RATE   = 2e-4
WARMUP_RATIO    = 0.05
LR_SCHEDULER    = "cosine"
WEIGHT_DECAY    = 0.01

# ── LoRA hyperparameters ─────────────────────────────────────────────────────
LORA_R          = 16       # rank
LORA_ALPHA      = 32       # scaling factor = alpha/r → 2×
LORA_DROPOUT    = 0
LORA_TARGETS    = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

# ── Evaluation/Split ─────────────────────────────────────────────────────────
EVAL_SPLIT      = 0.05     # 5% of data for validation
SEED            = 42

# ── HuggingFace token (set as Kaggle Secret named HF_TOKEN) ─────────────────
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
except ImportError:
    # Fallback to local environment variables if not running on Kaggle
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    print("⚠️  HF_TOKEN not set — model will not be pushed to Hub")
else:
    print("✅ HF_TOKEN found")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n📋 Config summary:")
print(f"   Base model : {BASE_MODEL_ID}")
print(f"   LoRA rank  : r={LORA_R}, alpha={LORA_ALPHA}")
print(f"   Epochs     : {EPOCHS}")
print(f"   Batch size : {BATCH_SIZE} × grad_accum={GRAD_ACCUM} × {NUM_GPUS} GPU(s)")

## 2️⃣ Dataset Loading & Exploration


In [ ]:
# ── Load Multi-Task Corpus ───────────────────────────────────────────────────
import os
from datasets import load_dataset
import pandas as pd

# Determine dataset path (local workspace vs Kaggle Input)
dataset_path = "corpus_VietMix_QA/corpus.jsonl"
if not os.path.exists(dataset_path):
    # Search in Kaggle inputs
    kaggle_input_dir = "/kaggle/input"
    found = False
    if os.path.exists(kaggle_input_dir):
        for root, dirs, files in os.walk(kaggle_input_dir):
            for file in files:
                if file == "corpus.jsonl":
                    dataset_path = os.path.join(root, file)
                    found = True
                    break
            if found:
                break

print(f"📥 Loading multitask dataset from: {dataset_path}")
raw_dataset = load_dataset("json", data_files=dataset_path, split="train")
print("\n📊 Dataset info:")
print(raw_dataset)

# Preview first 5 examples
df_preview = raw_dataset.to_pandas().head(5)
print(f"\n📋 First 5 examples:")
pd.set_option('display.max_colwidth', 120)
print(df_preview)

In [ ]:
# ── Explore Task Distribution ────────────────────────────────────────────────
df = raw_dataset.to_pandas()
print(f"📊 Total examples: {len(df):,}")
print("\n🔀 Distribution by task:")
print(df["task"].value_counts())

print("\n🔍 Sample Semantic Alignment example:")
sample_sa = df[df["task"] == "semantic_alignment"].iloc[0]
print(f"  [Input] : {sample_sa['input']}")
print(f"  [Output]: {sample_sa['output']}")

print("\n🔍 Sample Domain Q&A example:")
sample_qa = df[df["task"] == "domain_qa"].iloc[0]
print(f"  [Input] : {sample_qa['input']}")
print(f"  [Output]: {sample_qa['output']}")

## 3️⃣ Data Preprocessing — Build Instruction-Tuning Prompts


In [ ]:
# ── Verify dataset schema ────────────────────────────────────────────────────
REQUIRED_FIELDS = ["task", "instruction", "input", "output"]
missing = [f for f in REQUIRED_FIELDS if f not in raw_dataset.column_names]
if missing:
    raise ValueError(f"Dataset missing required columns: {missing}")
else:
    print("✅ All required columns present: task, instruction, input, output")

In [ ]:
# ── Define Multi-Task Prompt Formatting ──────────────────────────────────────
# Using ChatML prompt format to train the model to follow instructions

def format_record(instruction: str, input_text: str, output_text: str = None) -> dict:
    """Format a single record (instruction, input, output) using ChatML template."""
    text = (
        f"<|im_start|>system\n{instruction}<|im_end|>\n"
        f"<|im_start|>user\n{input_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    if output_text is not None:
        text += f"{output_text}<|im_end|"
    return {"text": text}

# Test formatting on a sample row
sample_row = raw_dataset[0]
sample_formatted = format_record(
    instruction=sample_row["instruction"],
    input_text=sample_row["input"],
    output_text=sample_row["output"]
)
print("📋 Sample formatted training prompt:")
print("─" * 60)
print(sample_formatted["text"])
print("─" * 60)

In [ ]:
# ── Build Instruction-Tuning Dataset ─────────────────────────────────────────
from datasets import Dataset

def build_instruction_dataset(raw_ds) -> Dataset:
    """Convert raw dataset examples into ChatML training examples."""
    examples = []
    for row in raw_ds:
        inst = str(row.get("instruction") or "").strip()
        inp  = str(row.get("input") or "").strip()
        out  = str(row.get("output") or "").strip()
        
        if not inp or not out:
            continue
            
        examples.append(format_record(inst, inp, out))
    return Dataset.from_list(examples)

print("⚙️  Splitting dataset into train and validation...")
# Split the raw data first so we can keep track of raw values during evaluation
raw_split = raw_dataset.train_test_split(test_size=EVAL_SPLIT, seed=SEED)
raw_train = raw_split["train"]
raw_eval  = raw_split["test"]

train_dataset = build_instruction_dataset(raw_train)
eval_dataset  = build_instruction_dataset(raw_eval)

print(f"\n📊 Dataset Split Summary:")
print(f"   Train size: {len(train_dataset):,} examples")
print(f"   Eval size : {len(eval_dataset):,} examples")

# Length analysis
import numpy as np
sample_lengths = [len(ex["text"].split()) for ex in train_dataset.select(range(min(1000, len(train_dataset))))]
print(f"\n📏 Approx word length stats:")
print(f"   Mean   : {np.mean(sample_lengths):.0f}")
print(f"   Median : {np.median(sample_lengths):.0f}")
print(f"   Max    : {np.max(sample_lengths)}")
print(f"   p95    : {np.percentile(sample_lengths, 95):.0f}")

## 4️⃣ Load Base Model with 4-bit Quantization


In [ ]:
from unsloth import FastLanguageModel

print(f"📥 Loading base model: {BASE_MODEL_ID}")
print("   (4-bit quantization via bitsandbytes — reduces VRAM by ~75%)")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto-detect (bfloat16 on Ampere+, float16 on T4)
    load_in_4bit=True,   # QLoRA: 4-bit base model
    token=HF_TOKEN if HF_TOKEN else None,
)

print(f"\n✅ Model loaded successfully!")
print(f"   Architecture  : {model.config.model_type}")
print(f"   Hidden size   : {model.config.hidden_size}")
print(f"   Num layers    : {model.config.num_hidden_layers}")
print(f"   Vocab size    : {model.config.vocab_size:,}")

# VRAM usage after loading
for i in range(NUM_GPUS):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    reserved  = torch.cuda.memory_reserved(i) / 1e9
    print(f"\n   GPU {i} VRAM: {allocated:.2f} GB allocated / {reserved:.2f} GB reserved")

## 5️⃣ Apply LoRA Adapters


In [ ]:
# ── Apply LoRA with UnSloth ───────────────────────────────────────────────────
# LoRA adds small trainable rank-decomposition matrices to attention layers
# Only ~1-5% of parameters are trained → fast & memory-efficient

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGETS,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",            # no bias tuning
    use_gradient_checkpointing="unsloth",  # UnSloth's memory-efficient checkpointing
    random_state=SEED,
    use_rslora=True,        # Rank-Stabilized LoRA (better stability)
    loftq_config=None,
)

# Count trainable parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"📊 Parameter summary:")
print(f"   Total params     : {total_params:,}")
print(f"   Trainable params : {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"   Frozen params    : {total_params - trainable_params:,}")

## 6️⃣ Training with SFTTrainer


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# ── Training arguments configuration ─────────────────────────────────────────
training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    
    # Batch & gradient configuration
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    
    # Learning rate schedule
    learning_rate = LEARNING_RATE,
    lr_scheduler_type = LR_SCHEDULER,
    warmup_steps = WARMUP_RATIO,
    weight_decay = WEIGHT_DECAY,
    
    # Epochs & strict matching strategy mapping
    num_train_epochs = EPOCHS,
    eval_strategy = "epoch",          # Updated from evaluation_strategy to match v5+ specifications
    save_strategy = "no",          # Must strictly match eval_strategy since load_best_model_at_end is active
    load_best_model_at_end = False,
    metric_for_best_model = "eval_loss",
    
    # Precision configuration targeting Tesla T4 boundaries
    fp16 = not is_bfloat16_supported(), # Will fallback safely to float16 execution on T4
    bf16 = is_bfloat16_supported(),
    
    # Logging and infrastructure
    logging_steps = 10,
    report_to = "none",
    seed = SEED,
    optim = "adamw_8bit",              # Utilizing 8-bit quantization memory saving features
    max_grad_norm = 0.3,
    push_to_hub = False,

    torch_empty_cache_steps = 20, # Cứ sau 20 steps sẽ tự động clear cache giải phóng VRAM rác
)

print("✅ TrainingArguments successfully aligned and configured.")
eff_batch: int = BATCH_SIZE * GRAD_ACCUM * NUM_GPUS
print(f"   Effective batch size calculated: {eff_batch}")
steps_per_epoch: int = len(train_dataset) // eff_batch
print(f"   Steps per epoch    : {steps_per_epoch:,}")
print(f"   Total target steps : {steps_per_epoch * EPOCHS:,}")

In [ ]:
# ── Build SFTTrainer ─────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,         
    args=training_args,
)

# Memory check before training
print("📊 VRAM usage before training:")
for i in range(NUM_GPUS):
    used = torch.cuda.memory_allocated(i) / 1e9
    total_vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   GPU {i}: {used:.2f} / {total_vram:.1f} GB ({100*used/total_vram:.1f}%)")

In [ ]:
# ── Start Training ───────────────────────────────────────────────────────────
import time

print("🚀 Starting LoRA fine-tuning...")
print(f"   This will take approximately {steps_per_epoch * EPOCHS // 60 + 1} minutes on 2×T4")
print("─" * 60)

start_time = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start_time

print("\n✅ Training complete!")
print(f"   Total time      : {elapsed/60:.1f} minutes")
print(f"   Train loss      : {trainer_stats.training_loss:.4f}")
print(f"   Steps completed : {trainer_stats.global_step}")

## 7️⃣ Evaluation — BLEU & chrF Scores


In [ ]:
# ── Evaluate Model on Both Tasks ──────────────────────────────────────────────
import evaluate
from unsloth import FastLanguageModel
import torch

# Switch to inference mode
FastLanguageModel.for_inference(model)

# Load translation metrics for Task 1
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def generate_response(instruction: str, input_text: str, max_new_tokens: int = 256) -> str:
    """Generate response for a single prompt using the model."""
    prompt = format_record(instruction, input_text)["text"]
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy decoding for deterministic eval
            pad_token_id=tokenizer.eos_token_id,
        )
    
    input_len = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return response

# Separate evaluation samples
eval_sa_rows = [row for row in raw_eval if row["task"] == "semantic_alignment"]
eval_qa_rows = [row for row in raw_eval if row["task"] == "domain_qa"]

print(f"📊 Evaluation Set: {len(eval_sa_rows)} Semantic Alignment, {len(eval_qa_rows)} Domain Q&A")

# 1. Evaluate Semantic Alignment (Task 1) using BLEU/chrF
print("\n🔤 Evaluating Task 1: Semantic Alignment (Translation)...")
sa_preds = []
sa_refs = []
for idx, row in enumerate(eval_sa_rows[:100]):  # evaluate up to 100 examples for speed
    pred = generate_response(row["instruction"], row["input"])
    sa_preds.append(pred)
    sa_refs.append([row["output"]])

if sa_preds:
    bleu_result = bleu_metric.compute(predictions=sa_preds, references=sa_refs)
    chrf_result = chrf_metric.compute(predictions=sa_preds, references=sa_refs)
    print(f"   BLEU Score : {bleu_result['score']:.2f}")
    print(f"   chrF Score : {chrf_result['score']:.2f}")

# 2. Evaluate Domain Q&A (Task 2) qualitatively
print("\n🎓 Qualitative Evaluation: Task 2: Domain Q&A...")
for idx, row in enumerate(eval_qa_rows[:5]):  # show first 5 examples
    pred = generate_response(row["instruction"], row["input"])
    print(f"\n[{idx+1}] Question: {row['input']}")
    print(f"    Reference Answer: {row['output']}")
    print(f"    Model Generated : {pred}")

## 8️⃣ Save & Export Model


In [ ]:
# ── Save LoRA adapters (lightweight — only ~100-500 MB) ──────────────────────
LORA_SAVE_PATH = f"{OUTPUT_DIR}/lora_adapters"
model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f"✅ LoRA adapters saved to: {LORA_SAVE_PATH}")

# List saved files
import os
saved_files = os.listdir(LORA_SAVE_PATH)
total_size = sum(os.path.getsize(f"{LORA_SAVE_PATH}/{f}") for f in saved_files)
print(f"   Saved {len(saved_files)} files | Total size: {total_size / 1e6:.1f} MB")

In [ ]:
# ── Option A: Save merged model (full precision) ─────────────────────────────
# This merges LoRA weights into the base model for standalone inference
# WARNING: Requires ~14 GB disk space for 7B model

MERGED_SAVE_PATH = f"{OUTPUT_DIR}/merged_model"
print(f"💾 Merging & saving full model to {MERGED_SAVE_PATH}...")
print("   (This may take 5-10 minutes)")

model.save_pretrained_merged(
    MERGED_SAVE_PATH,
    tokenizer,
    save_method="merged_16bit",  # merged float16 (half the size of float32)
)
print("✅ Merged model saved!")

In [ ]:
# ── Option B: Push LoRA adapters to HuggingFace Hub ──────────────────────────
if HF_TOKEN and HF_REPO_ID == "Khang3004/edumind-vietmix-qwen2.5-7b-lora":
    from huggingface_hub import login
    login(token=HF_TOKEN)
    
    print(f"📤 Pushing LoRA adapters to Hub: {HF_REPO_ID}")
    model.push_to_hub_merged(
        HF_REPO_ID,
        tokenizer,
        save_method="lora",
        token=HF_TOKEN,
    )
    print(f"✅ Uploaded to: https://huggingface.co/{HF_REPO_ID}")
else:
    print("⚠️  Skipping Hub push (HF_TOKEN not set or HF_REPO_ID not configured)")
    print("   To push: set HF_TOKEN as Kaggle Secret and update HF_REPO_ID in cell 3")

In [ ]:
# ── Option C: Export to GGUF (for llama.cpp / Ollama local deployment) ───────
# Useful if you want to run the model locally without GPU

GGUF_SAVE_PATH = f"{OUTPUT_DIR}/gguf_model"

print("📦 Exporting to GGUF (Q4_K_M quantization)...")
model.save_pretrained_gguf(
    GGUF_SAVE_PATH,
    tokenizer,
    quantization_method="q4_k_m",  # balanced quality/size
)

gguf_files = [f for f in os.listdir(GGUF_SAVE_PATH) if f.endswith(".gguf")]
for f in gguf_files:
    size_gb = os.path.getsize(f"{GGUF_SAVE_PATH}/{f}") / 1e9
    print(f"✅ GGUF file: {f} ({size_gb:.2f} GB)")

## 9️⃣ Quick Inference Test

> Run this cell anytime to test the fine-tuned model


In [ ]:
# ── Quick inference demo ─────────────────────────────────────────────────────
FastLanguageModel.for_inference(model)  # ensure inference mode

TEST_PROMPTS = [
    # Task 1: Semantic Alignment
    {
        "task": "semantic_alignment",
        "instruction": "Dịch câu tiếng Việt code-mixed AI/Data Science sau sang tiếng Anh học thuật chuẩn, giữ nguyên các thuật ngữ kỹ thuật và ý nghĩa chuyên môn.",
        "input": "Em đang dùng SFTTrainer của TRL để fine-tune, nhưng không hiểu tại sao dataset_text_field không tự động split."
    },
    {
        "task": "semantic_alignment",
        "instruction": "Dịch câu tiếng Việt code-mixed AI/Data Science sau sang tiếng Anh học thuật chuẩn, giữ nguyên các thuật ngữ kỹ thuật và ý nghĩa chuyên môn.",
        "input": "Anh ơi, em chạy model bị OOM trên con T4, có cách nào optimize VRAM bằng bitsandbytes không?"
    },
    # Task 2: Domain Q&A
    {
        "task": "domain_qa",
        "instruction": "Bạn là một trợ giảng AI/Data Science thông thái. Hãy giải đáp thắc mắc của học viên một cách ngắn gọn, chính xác bằng văn phong kỹ thuật tự nhiên.",
        "input": "Em bị dính lỗi OOM khi tuning con LLM bằng LoRA, có cách nào optimize batch size không anh?"
    },
    {
        "task": "domain_qa",
        "instruction": "Bạn là một trợ giảng AI/Data Science thông thái. Hãy giải đáp thắc mắc của học viên một cách ngắn gọn, chính xác bằng văn phong kỹ thuật tự nhiên.",
        "input": "Làm sao để merge adapter LoRA vào model gốc và export sang định dạng GGUF vậy ạ?"
    }
]

print("🎯 EduMIND Multi-Task Inference Demo")
print("=" * 60)

for i, test in enumerate(TEST_PROMPTS, 1):
    pred = generate_response(test["instruction"], test["input"])
    print(f"\n[{i}] Task: {test['task']}")
    print(f"    Input  : {test['input']}")
    print(f"    Output : {pred}")

## 🔄 API Fallback Integration

> Shows how EduMIND uses this fine-tuned model with API fallback


In [ ]:
# ── EduMIND Integration Pattern ───────────────────────────────────────────────
# This shows the .env config pattern for using the fine-tuned model

ENV_TEMPLATE = """
# ── EduMIND Translation Configuration ──────────────────────────────────────
# Option 1: Use fine-tuned LoRA model (offline, best domain accuracy)
EDUMIND_TRANSLATION_PROVIDER=huggingface
EDUMIND_TRANSLATION_MODEL=YOUR_HF_USERNAME/edumind-vietmix-qwen2.5-7b-lora
# or local path:
# EDUMIND_TRANSLATION_MODEL=/path/to/edumind-vietmix-llm/lora_adapters

# Option 2: API fallback (when no fine-tuned model available)
# EDUMIND_TRANSLATION_PROVIDER=gemini
# GOOGLE_API_KEY=your_key_here

# Option 3: OpenAI fallback
# EDUMIND_TRANSLATION_PROVIDER=openai
# OPENAI_API_KEY=your_key_here
"""

print("📋 EduMIND .env configuration template:")
print(ENV_TEMPLATE)
print("\n✅ After fine-tuning:")
print(f"   1. Update EDUMIND_TRANSLATION_MODEL in your .env")
print(f"   2. Set EDUMIND_TRANSLATION_PROVIDER=huggingface")
print(f"   3. Restart the EduMIND service")
print(f"   4. The HuggingFaceTranslationProvider will load your fine-tuned model")